In [1]:
import os
import torch
import pandas as pd

from tqdm import tqdm

from stock_gpt import StockGPT, LinearModel, NaiveModel
from dataloader_builder import build_dataloaders
from setup import StockGPT_cfg, LinearModel_cfg, NaiveModel_cfg
from setup import path_data_preprocessor, PATH_RESULTS_NON_RESIDUALS, PATH_RESULTS_RESIDUALS
from model_training import model_setup, train_model_cuda

from model_training import train_model_cuda, evaluate_model, evaluate_best_model
from model_analysis import test_model, print_loss_analysis, process_losses, format_num, process_result, store_result

In [2]:
cuda = True if torch.cuda.is_available() else False

print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PyTorch: 2.13.0+cu132
CUDA build: 13.2
CUDA available: True
GPU: NVIDIA GeForce RTX 4070 Laptop GPU


## MODEL TRAINING ---------------------------

In [3]:
torch.manual_seed(1234)
dls, train_norms = build_dataloaders(path_data_preprocessor)

Building DataLoaders...


In [ ]:
optimizer_data = [torch.optim.AdamW, 0.0004, 0.1]
scaler_data = [torch.amp.GradScaler, "cuda"]

max_epochs = 15

eval_bs = 1000

stockGPT, stockGPT_params, opt1, sca1, sch1 = model_setup(StockGPT, StockGPT_cfg, train_norms, device,
                                                *optimizer_data, *scaler_data)
linearModel, linearModel_params, opt2, sca2, sch2 = model_setup(LinearModel, LinearModel_cfg, train_norms, device, 
                                                      *optimizer_data, *scaler_data)
naiveModel = NaiveModel(NaiveModel_cfg, train_norms)
naiveModel.to(device)

model_train_losses, model_val_losses = train_model_cuda(stockGPT, device, opt1, sca1, sch1, max_epochs, 
                                                        dls["train"], dls["val"], eval_bs)
linear_train_losses, linear_val_losses = train_model_cuda(linearModel, device, opt2, sca2, sch2, max_epochs,
                                                        dls["train"], dls["val"], eval_bs)


Input Norm: torch.Size([12])|torch.Size([12])
Target Norm: torch.Size([4])|torch.Size([4])
3261440
5376
Continuing from previous checkpoint...


|          | 0.0% (00:00) Setting up...                                                                   

Epoch 12:

Learning Rate: 4.00e-04



|██▌       | 25.0% (19:54) Evaluating model on validation data... (101/102) [1874/7496]:                  

Epoch 12:
Training Loss:
   (MAE) 0.002549779834225774
   (NLL) -3.7459828853607178
Validation Loss:
   (MAE) 0.002494317479431629
   (NLL) -3.7563579082489014

Best Validation: -4.029028415679932
----------------------------------------------------------------------------------------------------

Learning Rate: 4.00e-04



|█████     | 50.0% (43:29) Evaluating model on validation data... (101/102) [3748/7496]: 

Epoch 13:
Training Loss:
   (MAE) 0.001999286701902747
   (NLL) -3.9335827827453613
Validation Loss:
   (MAE) 0.001993885263800621
   (NLL) -3.9397289752960205

Best Validation: -4.029028415679932
----------------------------------------------------------------------------------------------------

Learning Rate: 2.00e-04



|███████▌  | 75.0% (1:05:54) Evaluating model on validation data... (101/102) [5622/7496]: 

Epoch 14:
Training Loss:
   (MAE) 0.002214641310274601
   (NLL) -4.471590995788574
Validation Loss:
   (MAE) 0.002230408601462841
   (NLL) -4.488435745239258

Best Validation: -4.488435745239258
----------------------------------------------------------------------------------------------------

Learning Rate: 2.00e-04



|████████▏ | 82.2% (1:23:53) Training StockGPT-B1... [6160/7496]:                          

## Model Analysis -------------------------

In [ ]:
#* REUSES OBJETCS FROM TRAINING
analysis_steps = min(eval_bs, len(dls["train"])) + min(eval_bs, len(dls["val"])) + min(eval_bs, len(dls["test"]))
analysis_pbar = tqdm(total=3*analysis_steps, desc=f"Evaluating the best model parameters...".ljust(80),
                bar_format="|{bar}| {percentage:3.1f}% ({elapsed}) {desc}", position=0, leave=False)

#* Reevaluates models by their best parameters on train and val dataloaders
naive_losses = evaluate_model(dls["train"], dls["val"], naiveModel, device, eval_bs, analysis_pbar)
linear_losses = evaluate_best_model(linearModel, device, opt2, sca2, sch2, dls["train"], dls["val"], eval_bs, analysis_pbar, True)
gpt_losses = evaluate_best_model(stockGPT, device, opt1, sca1, sch1, dls["train"], dls["val"], eval_bs, analysis_pbar, True) 

#* Final evaluation on unseen test dataloader
naive_test_losses = test_model(dls["test"], naiveModel, device, eval_bs, analysis_pbar)
linear_test_losses = test_model(dls["test"], linearModel, device, eval_bs, analysis_pbar)
gpt_test_losses = test_model(dls["test"], stockGPT, device, eval_bs, analysis_pbar)


|██████████| 100.0% (05:06) Evaluating model on testing data... (36/37) [2214/2214]:                      

In [ ]:
for key, features in [("NLL", StockGPT_cfg["target_features"]),
                      ("STD", [f"{feature}_std" for feature in StockGPT_cfg["target_features"]]),
                      ("MAE", StockGPT_cfg["target_features"]),
                      ("PMAE", StockGPT_cfg["target_features"])]:
    print_loss_analysis(process_losses(gpt_losses + gpt_test_losses +
                                       linear_losses + linear_test_losses +
                                       naive_losses + naive_test_losses, key), 
                                       [stockGPT.cfg["name"], linearModel.cfg["name"], naiveModel.cfg["name"]],
                                       [format_num(stockGPT_params), format_num(linearModel_params), "0"], 
                                       features, key)


--------------------------------------------------------------------------------------------------------------

NLL

--------------------------------------------------------------------------------------------------------------

                    o        h        l        c        
StockGPT-B1: 3.3M
    Training:       -4.0318  -4.0284  -3.7748  -4.2336    >  -4.0171
    Validation:     -4.0459  -4.0334  -3.7850  -4.2518    >  -4.0290
    Testing:        -4.0797  -4.0769  -3.8098  -4.2986    >  -4.0663
    
LinearModel-B1: 5.4K
    Training:       0.9568   13.6889  2.5214   12.7406    >  7.4769
    Validation:     0.9024   12.8702  2.2666   15.5718    >  7.9028
    Testing:        0.9050   12.5521  2.2168   15.4411    >  7.7788
    
NaiveModel-B1: 0
    Training:       2.5890   2.5894   2.5886   2.5890     >  2.5890
    Validation:     2.6175   2.6175   2.6174   2.6174     >  2.6174
    Testing:        2.6858   2.6857   2.6860   2.6858     >  2.6858
    

--------------------------

In [ ]:
import importlib
import setup
importlib.reload(setup)
from setup import PATH_RESULTS_RESIDUALS

store_result(PATH_RESULTS_RESIDUALS, process_result(stockGPT, gpt_losses, gpt_test_losses, max_epochs))
store_result(PATH_RESULTS_RESIDUALS, process_result(linearModel, linear_losses, linear_test_losses, max_epochs))
store_result(PATH_RESULTS_RESIDUALS, process_result(naiveModel, naive_losses, naive_test_losses, max_epochs))

print(pd.read_parquet(PATH_RESULTS_RESIDUALS))

{'model': 'StockGPT-B1', 'bar_width': 1, 'train': {'NLL': [-4.031753063201904, -4.028439044952393, -3.774773597717285, -4.233578205108643], 'STD': [0.006745090242475271, 0.006522008217871189, 0.00906664039939642, 0.005418163724243641], 'MAE': [0.00223139557056129, 0.0016865597572177649, 0.0016007142839953303, 0.0015706036938354373], 'PMAE': [75004240.0, 44459996.0, 36925412.0, 36088352.0]}, 'val': {'NLL': [-4.045901775360107, -4.033441543579102, -3.784982681274414, -4.251786708831787], 'STD': [0.006761156488209963, 0.006579043343663216, 0.009069887921214104, 0.005451349075883627], 'MAE': [0.0022572213783860207, 0.0016836533322930336, 0.0015666458057239652, 0.0015233720187097788], 'PMAE': [96096400.0, 57985784.0, 47308508.0, 48474012.0]}, 'test': {'NLL': [-4.079662799835205, -4.076940059661865, -3.8098065853118896, -4.298624038696289], 'STD': [0.00634319894015789, 0.0063157049007713795, 0.008658348582684994, 0.005052647553384304], 'MAE': [0.0018326368881389499, 0.001258758013136685, 0.0

In [ ]:
from data_scrapper import scrape_data, get_all_tickers
from data_filler import fill_data
from data_preprocessor import preprocess_data
from setup import API_KEY, TIMEFRAME
import pandas_market_calendars as mcal

In [ ]:
return ""
all_tickers = get_all_tickers("raw_data/all_tickers_trimmed_1_30", API_KEY)
scrape_data(API_KEY,
              "raw_data/data_5min_2026",
              250,
              all_tickers,
              mcal.get_calendar("NYSE").schedule("2026-01-01","2026-7-1").index,
              TIMEFRAME)
fill_data("raw_data/data_5min_2026",
          "filled_raw_data/data_5min_2026",
          mcal.get_calendar("NYSE").schedule("2026-01-01","2026-7-1").index)
preprocess_data("filled_raw_data/data_5min_2026",
                "preprocessed_data/data_5min_2026",
                mcal.get_calendar("NYSE").schedule("2026-01-01","2026-7-1").index, [0.75, 0.9])

SyntaxError: 'return' outside function (521317128.py, line 1)

In [ ]:
dls, train_norms = build_dataloaders("preprocessed_data/data_1min_2026")

Building DataLoaders...


In [ ]:
test_losses = test_model(dls["test"], stockGPT, device, eval_bs)
print(test_losses)

({'NLL': tensor([-4.0558, -4.0586, -3.7922, -4.2680], device='cuda:0'), 'STD': tensor([0.0064, 0.0063, 0.0088, 0.0051], device='cuda:0'), 'MAE': tensor([0.0019, 0.0014, 0.0013, 0.0013], device='cuda:0'), 'PMAE': tensor([72077192., 39803644., 31995208., 32325556.], device='cuda:0')},)
